# Identify T12 training files in a source folder

For each Excel file in `SOURCE_DIR` (recursively), decide one of:

| Bucket | Meaning | Action |
|---|---|---|
| `mapped` | Has a `Financials` sheet with a **"COMPLETE MAPPING HERE"** anchor row | → import to `data/mapped_T12_docs/` |
| `broken` | `.xls` file (openpyxl can't read) or open failure | → manually re-save as `.xlsx` in Excel |
| `other` | No `Financials` sheet or no anchor — not a training file | → skip |

The classifier mirrors `extract_reviewed_row_history.py` exactly — both `.xlsx` and `.xlsm` count as training files as long as the structure is there.

**Why we have to iterate the entire sheet:** deal-model workbooks have huge `Financials` sheets (5000+ rows × 200+ cols). The "COMPLETE MAPPING HERE" anchor sits in the *middle* of the sheet, not near the top. An earlier shallow-scan version of this notebook bailed at row 300 and missed 17 valid training files.

Outputs:

1. Summary counts by bucket
2. Mapped files grouped by source subfolder, marked `*` if already in repo, with mapped-row counts
3. Broken files (full paths, so you can find and convert them in Excel)
4. Files to skip (just names, for visibility)
5. Dry-run copy cell — flip `DRY_RUN = False` and re-run to actually import the new files

In [1]:
from pathlib import Path
import openpyxl

# ── Tweak this ────────────────────────────────────────────────────────────────
SOURCE_DIR = Path(
    "/Users/manikantamb/_MyFiles/Work_/Freelancing/Shivam_Excel_AI_Automation/"
    "master_docs/client_sent_docs/6th_june_new_t12_docs/T12"
)

# Where the rulebook training set lives in the repo (for dedup checks)
REPO_DATA = Path(
    "/Users/manikantamb/_MyFiles/Work_/Freelancing/Shivam_Excel_AI_Automation/"
    "app/t12_pipeline/data"
)
MAPPED_T12_DOCS_DIR = REPO_DATA / "mapped_T12_docs"

EXISTING_TRAINING = {
    p.name for p in MAPPED_T12_DOCS_DIR.glob("*.xls*")
    if not p.name.startswith("~$")
}

print(f"SOURCE_DIR:          {SOURCE_DIR}")
print(f"Training set in repo: {MAPPED_T12_DOCS_DIR}")
print(f"Files already in repo: {len(EXISTING_TRAINING)}")

SOURCE_DIR:          /Users/manikantamb/_MyFiles/Work_/Freelancing/Shivam_Excel_AI_Automation/master_docs/client_sent_docs/6th_june_new_t12_docs/T12
Training set in repo: /Users/manikantamb/_MyFiles/Work_/Freelancing/Shivam_Excel_AI_Automation/app/t12_pipeline/data/mapped_T12_docs
Files already in repo: 24


## Classifier + row-counter

`classify_file(path)` returns a dict like:

```python
{
    "bucket": "mapped" | "broken" | "other",
    "reason": str,                # human-readable explanation
    "anchor_row": int | None,     # where "COMPLETE MAPPING HERE" was found
    "mapped_rows": int | None,    # how many populated rows the file has
}
```

`mapped_rows` lets us spot files where the structure is there but no actual mapping data exists (e.g. empty templates).

In [2]:
REQUIRED_COLS = {"Row Description (from T12)", "Auto Mapping", "OVERRIDE", "Final_Mapping"}


def _find_anchor_row(ws) -> int | None:
    """Return row index where 'COMPLETE MAPPING HERE' lives, or None. Scans the whole sheet."""
    for i, row in enumerate(ws.iter_rows(values_only=True), 1):
        for cell in row:
            if cell and isinstance(cell, str) and "COMPLETE MAPPING HERE" in cell.upper():
                return i
    return None


def _count_mapped_rows(ws, anchor_row: int) -> int:
    """Count rows below the anchor with a non-empty Row Description and a mapping value."""
    headers = None
    header_row_idx = None
    for i, row in enumerate(ws.iter_rows(min_row=anchor_row + 1, max_row=anchor_row + 10, values_only=True),
                            anchor_row + 1):
        cells = [str(c).strip() if c is not None else "" for c in row]
        if "Row Description (from T12)" in cells:
            headers = cells
            header_row_idx = i
            break
    if not headers:
        return 0

    desc_col = headers.index("Row Description (from T12)")
    fm_col = headers.index("Final_Mapping") if "Final_Mapping" in headers else None
    ov_col = headers.index("OVERRIDE") if "OVERRIDE" in headers else None

    count = 0
    for row in ws.iter_rows(min_row=header_row_idx + 1, max_row=header_row_idx + 500, values_only=True):
        desc = row[desc_col] if desc_col < len(row) else None
        if not desc or not isinstance(desc, str) or not desc.strip():
            continue
        fm = row[fm_col] if fm_col is not None and fm_col < len(row) else None
        ov = row[ov_col] if ov_col is not None and ov_col < len(row) else None
        has_map = (
            (fm and str(fm).strip() and str(fm).strip() != "#NAME?") or
            (ov and str(ov).strip())
        )
        if has_map:
            count += 1
    return count


def classify_file(path: Path) -> dict:
    """Decide if a file is a usable T12 training file. Mirrors extract_reviewed_row_history.py."""
    if path.suffix.lower() == ".xls":
        return {
            "bucket": "broken", "reason": "old .xls — manually convert to .xlsx in Excel",
            "anchor_row": None, "mapped_rows": None,
        }
    try:
        wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    except Exception as e:
        return {
            "bucket": "broken", "reason": f"open failed: {str(e)[:60]}",
            "anchor_row": None, "mapped_rows": None,
        }

    if "Financials" not in wb.sheetnames:
        wb.close()
        return {
            "bucket": "other", "reason": "no 'Financials' sheet",
            "anchor_row": None, "mapped_rows": None,
        }

    ws = wb["Financials"]
    anchor = _find_anchor_row(ws)
    if anchor is None:
        wb.close()
        return {
            "bucket": "other", "reason": "no 'COMPLETE MAPPING HERE' anchor",
            "anchor_row": None, "mapped_rows": None,
        }

    mapped_rows = _count_mapped_rows(ws, anchor)
    wb.close()
    return {
        "bucket": "mapped", "reason": "OK",
        "anchor_row": anchor, "mapped_rows": mapped_rows,
    }

## Walk + classify everything

Records the source-relative subfolder for each file so we can group by it later.

In [3]:
EXCEL = {".xlsx", ".xlsm", ".xls"}

results = []  # list of (path, subfolder, result_dict)
for p in sorted(SOURCE_DIR.rglob("*")):
    if not p.is_file() or p.suffix.lower() not in EXCEL:
        continue
    if p.name.startswith("~$") or p.name == ".DS_Store":
        continue
    subfolder = str(p.parent.relative_to(SOURCE_DIR)) or "(root)"
    res = classify_file(p)
    results.append((p, subfolder, res))

# Bucket-by-bucket summary
print(f"{'Bucket':<10} Count")
print("-" * 18)
for bucket in ("mapped", "broken", "other"):
    n = sum(1 for _, _, r in results if r["bucket"] == bucket)
    print(f"{bucket:<10} {n}")
print(f"{'TOTAL':<10} {len(results)}")

Bucket     Count
------------------
mapped     42
broken     2
other      44
TOTAL      88


/Users/manikantamb/Library/Python/3.14/lib/python/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Source Data'!$B$4:INDEX('Source Data'!$P$12:$P$268,COUNTA('Source Data'!$B$12:$B$268)).
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/Users/manikantamb/Library/Python/3.14/lib/python/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Source Data'!$B$4:INDEX('Source Data'!$P$12:$P$448,COUNTA('Source Data'!$B$12:$B$448)).
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/Users/manikantamb/Library/Python/3.14/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


## Mapped files — by source subfolder

`*` next to a name means **already in `data/mapped_T12_docs/`** (skip the copy).
The number is the count of populated mapped rows in the file.

In [4]:
from collections import defaultdict

by_folder = defaultdict(list)
for p, sub, r in results:
    if r["bucket"] == "mapped":
        by_folder[sub].append((p, r))

total_new = 0
total_dup = 0
for folder in sorted(by_folder):
    entries = sorted(by_folder[folder], key=lambda x: x[0].name)
    new_n = sum(1 for p, _ in entries if p.name not in EXISTING_TRAINING)
    dup_n = len(entries) - new_n
    total_new += new_n
    total_dup += dup_n

    print(f"\n── {folder}  ({len(entries)} files: {new_n} NEW, {dup_n} already in repo) ──")
    for p, r in entries:
        flag = "*" if p.name in EXISTING_TRAINING else " "
        rows = r["mapped_rows"]
        print(f"  {flag} [{rows:>4} rows]  {p.name}")

print(f"\n{'=' * 60}")
print(f"TOTAL across all subfolders: {total_new} NEW + {total_dup} already in repo")


── New  (17 files: 17 NEW, 0 already in repo) ──
    [ 205 rows]  02242026_Grande Reserve at Geist_V3- VM..xlsx
    [ 112 rows]  Arbor Gates at Buckhead Model T12.xlsx
    [ 142 rows]  Cardinal Model T12.xlsx
    [ 192 rows]  Creekside Apartment Homes Model T12.xlsx
    [ 117 rows]  Eleven11 Lexington at Flower Mound Model T12.xlsx
    [ 205 rows]  Grande Reserve at Geist Model T12.xlsx
    [ 118 rows]  Holden Model T12.xlsx
    [ 117 rows]  Ladera Model T12.xlsx
    [ 127 rows]  Lincoln Townhomes Model T12.xlsx
    [ 155 rows]  Lockhart Model T12.xlsx
    [ 148 rows]  Mesquite Model T12.xlsx
    [ 109 rows]  Mirador Model T12.xlsx
    [  92 rows]  Reserve_at_White_Rock Model T12.xlsx
    [ 113 rows]  River Point Model T12.xlsx
    [ 113 rows]  River Point West Model T12.xlsx
    [  87 rows]  TRAILS OF WHITE ROCK Model T12.xlsx
    [ 202 rows]  Wythe Model T12.xlsx

── Previously Provided  (25 files: 1 NEW, 24 already in repo) ──
  * [ 141 rows]  02242026_Grande Reserve at Geist_V3- V

## Broken files — need manual conversion to `.xlsx`

These are `.xls` files (or files openpyxl can't open). Open each one in Excel and **Save As → Excel Workbook (.xlsx)** at the same location, then re-run this notebook to pick them up.

In [5]:
broken = [(p, sub, r) for p, sub, r in results if r["bucket"] == "broken"]
if not broken:
    print("  (none)")
else:
    for p, sub, r in sorted(broken, key=lambda x: x[0].name):
        print(f"  • {p.name}")
        print(f"     in subfolder: {sub}")
        print(f"     reason:       {r['reason']}")
        print(f"     full path:    {p}")
        print()

  • RESERVE AT WHITE ROCK Seller T12.xls
     in subfolder: New
     reason:       old .xls — manually convert to .xlsx in Excel
     full path:    /Users/manikantamb/_MyFiles/Work_/Freelancing/Shivam_Excel_AI_Automation/master_docs/client_sent_docs/6th_june_new_t12_docs/T12/New/RESERVE AT WHITE ROCK Seller T12.xls

  • TRAILS AT WHITE ROCK Seller T12.xls
     in subfolder: New
     reason:       old .xls — manually convert to .xlsx in Excel
     full path:    /Users/manikantamb/_MyFiles/Work_/Freelancing/Shivam_Excel_AI_Automation/master_docs/client_sent_docs/6th_june_new_t12_docs/T12/New/TRAILS AT WHITE ROCK Seller T12.xls



## Files to skip (no mapping data)

For visibility only. These are rent rolls, raw vendor T12s without mapping, model outputs without the COMPLETE MAPPING anchor, etc.

In [6]:
others = [(p, sub) for p, sub, r in results if r["bucket"] == "other"]
print(f"Total to skip: {len(others)}\n")
for p, sub in sorted(others, key=lambda x: x[0].name):
    print(f"  [{sub}]  {p.name}")

Total to skip: 44

  [New]  Arbor Gates at Buckhead Seller T12.xlsx
  [Previously Provided]  Brownsburg Crossing T12 05-25.xlsx
  [New]  Cardinal Seller T12.xlsx
  [Previously Provided]  Carmel Woods T12 09-25.xlsx
  [Previously Provided]  Chateau in the  Woods T12 08-25.xlsx
  [Previously Provided]  Copy of Prescott Park - RR - 3.20.26.xlsx
  [New]  Creekside Seller T12.xlsx
  [New]  Crystal Lake_T-12_04.2026.xlsx
  [Previously Provided]  Devon Rent Roll 3.11.2026.xlsx
  [Previously Provided]  Echo Park at Perry Crossing T12 12-24.xlsx
  [New]  Eleven11 Lexington at Flower Mound Seller T12.xlsx
  [Previously Provided]  English Village RR 12-17-25.xlsx
  [Previously Provided]  Exchange 7272 - RR - 10.15.25.xlsx
  [Previously Provided]  Forma at the Park - RR - 3.2.26.xlsx
  [Previously Provided]  Golfview Manor RR 04-22-2024.xlsx
  [Previously Provided]  Golfview Manor RR 04-22-24 RedIQ.xlsx
  [Previously Provided]  Hangar RR 04-22-24 RedIQ.xlsx
  [Previously Provided]  Hangar RR 04-22

## Copy NEW training files into the repo

Copies every `mapped` file whose name isn't already in `data/mapped_T12_docs/`.

- `DRY_RUN = True` (default) → just prints what *would* be copied. Safe to run.
- Flip to `False` and re-run the cell to actually copy.

In [8]:
import shutil

DRY_RUN = False   # flip to False to actually copy

new_to_copy = [
    (p, r) for p, _, r in results
    if r["bucket"] == "mapped" and p.name not in EXISTING_TRAINING
]

prefix = "[DRY RUN] " if DRY_RUN else ""
print(f"{prefix}Will copy {len(new_to_copy)} new training file(s) -> {MAPPED_T12_DOCS_DIR}\n")
for src, r in sorted(new_to_copy, key=lambda x: x[0].name):
    dst = MAPPED_T12_DOCS_DIR / src.name
    print(f"  + [{r['mapped_rows']:>4} rows]  {src.name}")
    if not DRY_RUN:
        shutil.copy2(src, dst)

if DRY_RUN:
    print("\n(no files copied — set DRY_RUN = False and re-run to perform the copy)")
else:
    total_after = len(list(MAPPED_T12_DOCS_DIR.glob('*.xls*')))
    print(f"\nDone. Total in {MAPPED_T12_DOCS_DIR.name}/: {total_after}")

Will copy 18 new training file(s) -> /Users/manikantamb/_MyFiles/Work_/Freelancing/Shivam_Excel_AI_Automation/app/t12_pipeline/data/mapped_T12_docs

  + [ 205 rows]  02242026_Grande Reserve at Geist_V3- VM..xlsx
  + [ 124 rows]  100325_Park at Eagle Creek_v3.xlsx
  + [ 112 rows]  Arbor Gates at Buckhead Model T12.xlsx
  + [ 142 rows]  Cardinal Model T12.xlsx
  + [ 192 rows]  Creekside Apartment Homes Model T12.xlsx
  + [ 117 rows]  Eleven11 Lexington at Flower Mound Model T12.xlsx
  + [ 205 rows]  Grande Reserve at Geist Model T12.xlsx
  + [ 118 rows]  Holden Model T12.xlsx
  + [ 117 rows]  Ladera Model T12.xlsx
  + [ 127 rows]  Lincoln Townhomes Model T12.xlsx
  + [ 155 rows]  Lockhart Model T12.xlsx
  + [ 148 rows]  Mesquite Model T12.xlsx
  + [ 109 rows]  Mirador Model T12.xlsx
  + [  92 rows]  Reserve_at_White_Rock Model T12.xlsx
  + [ 113 rows]  River Point Model T12.xlsx
  + [ 113 rows]  River Point West Model T12.xlsx
  + [  87 rows]  TRAILS OF WHITE ROCK Model T12.xlsx
  + [ 20